# LC #76 — Minimum Window Substring
**Category:** Sliding Window + HashMap | **Difficulty:** Hard | **Day 2**

---

<div style="padding:15px;border-left:8px solid #11998e;background:#e8faf5;border-radius:4px;">
<strong>The Problem:</strong> Given strings <code>s</code> and <code>t</code>, return the minimum window
substring of <code>s</code> that contains all characters of <code>t</code>.
</div>

**Examples:**
```
s = "ADOBECODEBANC", t = "ABC" → "BANC"
s = "a",             t = "a"  → "a"
s = "a",             t = "aa" → ""  (can't satisfy)
```

**Core Insight:** Expand right until the window contains all required characters (valid). Then shrink left as much as possible while still valid. Record the minimum. Repeat.

## Mental Models

**1. The "Have / Required" Counter**
`required` = number of *distinct* characters in `t` that must be satisfied (each at the right count).
`have` = number of those characters currently satisfied in the window.
Window is valid when `have == required`.

**2. Satisfy vs Over-satisfy**
Satisfying means meeting exactly the required count — not exceeding it. If `t = "AA"`, the window must have at least 2 A's. Having 3 still satisfies (we only track when we reach *exactly* the needed count from below).

**3. Expand-then-Shrink Cycle**
Right expands until valid → record window → shrink left until invalid → right expands again. This two-pointer dance covers all candidate windows in O(n+m).

In [ ]:
# Brute Force — O(n² * m) where m = len(t)
from collections import Counter

def minWindow_brute(s: str, t: str) -> str:
    need = Counter(t)
    best = ""
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            window = Counter(s[i:j])
            # Check if window contains all of t
            if all(window[c] >= need[c] for c in need):
                if not best or j - i < len(best):
                    best = s[i:j]
    return best

print(minWindow_brute("ADOBECODEBANC", "ABC"))   # BANC
print(minWindow_brute("a", "a"))                  # a

## The Two-Pointer Strategy

```
s = "ADOBECODEBANC", t = "ABC"
need = {A:1, B:1, C:1}   required = 3

Phase 1 — Expand right until have == 3:
A→D→O→B→E→C  → window "ADOBEC", have=3 ✓

Phase 2 — Shrink left while have == 3:
Remove A → "DOBEC", have=2  ✗  →  record "ADOBEC" (len=6)

Phase 3 — Continue expanding right:
O→D→E→B → "DOBECODEB", have=2
→ A → "DOBECODEBAN", have=2
→ N → "DBECODEBANCB"...
→ C → "DOBECODEBANCB"... have=3 ✓

Shrink left:
D,O,B,E,C,O,D,E → "BANC" still has A,B,C → have=3 ✓ → record "BANC" (len=4) ✓
B removed → have=2 ✗

Final result: "BANC"
```

In [ ]:
# Optimal — O(n+m) time, O(m) space
from collections import Counter

def minWindow(s: str, t: str) -> str:
    if not t or not s:
        return ""

    need = Counter(t)          # required freq for each char in t
    have, required = 0, len(need)
    left = 0
    result = ""
    window = {}

    for right, char in enumerate(s):
        window[char] = window.get(char, 0) + 1
        # Check if adding char satisfied its requirement (exactly met, not exceeded)
        if char in need and window[char] == need[char]:
            have += 1

        # Shrink left as long as window is valid
        while have == required:
            window_size = right - left + 1
            if not result or window_size < len(result):
                result = s[left:right+1]
            # Remove leftmost char
            left_char = s[left]
            window[left_char] -= 1
            if left_char in need and window[left_char] < need[left_char]:
                have -= 1
            left += 1

    return result

# Test
print(minWindow("ADOBECODEBANC", "ABC"))   # BANC
print(minWindow("a", "a"))                  # a
print(minWindow("a", "aa"))                 # (empty — can't satisfy)
print(minWindow("aa", "aa"))               # aa

## Complexity Analysis

| Approach | Time | Space |
|----------|------|-------|
| Brute Force | O(n² × m) | O(m) |
| **Sliding Window** | **O(n + m)** | **O(m)** |

**O(n+m):** We scan `s` once (O(n)) and build `need` from `t` (O(m)). Each character in `s` is added and removed from the window at most once.

**O(m) space:** `need` holds |t| unique characters. `window` holds at most |s| keys in the worst case.

## Interview Q&A

**Q: What does `have` track exactly?**
A: The number of distinct characters from `t` whose count in the current window meets or exceeds the required count. When `have == required`, every character in `t` is satisfied.

**Q: Why check `window[char] == need[char]` (not `>=`)?**
A: We only increment `have` when we *first* reach the required count. If we already exceeded it, adding another copy doesn't increment `have` again (avoiding double-counting).

**Q: Why decrement `have` when `window[left_char] < need[left_char]`?**
A: When removing the left char drops its count *below* required, we've un-satisfied that character. `have` decrements to signal the window is no longer valid.

**Q: What's the general two-pointer template?**
A: Expand right to reach valid → record result → shrink left until invalid → repeat. This template works for minimum window, maximum window with constraint, etc.

## The Citi Angle

**The "minimum covering set" pattern:** Find the shortest time window that contains at least one occurrence of every required metric type.

```python
from collections import Counter

# Minimum time window containing all required alert types
event_stream = ["cpu","mem","cpu","disk","net","cpu","mem","disk"]
required_types = ["cpu", "mem", "disk"]

events = list(enumerate(event_stream))  # (index, type)
need = Counter(required_types)
have, req = 0, len(need)
left = 0
window_counts = {}
min_window = None

for right, etype in events:
    window_counts[etype] = window_counts.get(etype, 0) + 1
    if etype in need and window_counts[etype] == need[etype]:
        have += 1
    while have == req:
        if min_window is None or right - left < min_window[1] - min_window[0]:
            min_window = (left, right)
        ltype = event_stream[left]
        window_counts[ltype] -= 1
        if ltype in need and window_counts[ltype] < need[ltype]:
            have -= 1
        left += 1

print(f"Minimum window: indices {min_window[0]}-{min_window[1]}")
print(f"Events: {event_stream[min_window[0]:min_window[1]+1]}")
```

**Interview tie-in:** "Minimum window substring is the pattern for finding the shortest time period covering all required event types — a real diagnostic tool for root cause analysis."

## Summary

| | Value |
|--|--|
| **Pattern** | Sliding Window + HashMap (char counts) |
| **Time** | O(n + m) |
| **Space** | O(m) |
| **Key tracking** | `have` = satisfied chars; `required` = distinct chars in t |
| **Say in interview** | "Expand right until have==required (valid). Shrink left while still valid, recording minimum. have tracks distinct-char satisfaction, not total chars." |